In [ ]:
# 1. Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense

# 2. Load Dataset
data = pd.read_csv("smart_grid.csv")
print(data.shape)
print(data.head())

# 3. Separate Features and Target
X = data.drop(["Timestamp", "Anomaly_Label"], axis=1).values
y = data["Anomaly_Label"].values

# 4. Fill Missing Values
X = pd.DataFrame(X).fillna(pd.DataFrame(X).mean()).values

# 5. Normalize Data
X = MinMaxScaler().fit_transform(X)

# 6. Create Time Windows
def create_windows(data, size=10):
    return np.array([data[i:i+size] for i in range(len(data)-size+1)])

X = create_windows(X)
y = y[9:]

# 7. Select Normal Data
X_normal = X[y == 0]

# 8. Split Training and Testing Data
X_train = X_normal[:int(0.8 * len(X_normal))]
X_test = X[int(0.8 * len(X)):]
y_test = y[int(0.8 * len(y)):]

# 9. Build LSTM Autoencoder
model = Sequential([
    LSTM(64, activation="relu", input_shape=(10, 5)),
    RepeatVector(10),
    LSTM(64, activation="relu", return_sequences=True),
    TimeDistributed(Dense(5))
])

# 10. Compile Model
model.compile(optimizer="adam", loss="mse", metrics=["mse"])

# 11. Train Model
history = model.fit(
    X_train, X_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

# 12. Calculate Reconstruction Error
train_error = np.mean((X_train - model.predict(X_train))**2, axis=(1,2))
test_error = np.mean((X_test - model.predict(X_test))**2, axis=(1,2))

# 13. Find Threshold
threshold = np.mean(train_error) + np.std(train_error)
print("Threshold:", threshold)

# 14. Predict Anomalies
y_pred = (test_error > threshold).astype(int)

# 15. Calculate Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred) * 100, "%")
print("Actual Anomalies:", sum(y_test))
print("Predicted Anomalies:", sum(y_pred))

# 16. Plot Training Loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.show()

# 17. Plot Reconstruction Error
plt.hist(test_error, bins=50)
plt.axvline(threshold, color="red", linestyle="--")
plt.title("Anomaly Detection")
plt.xlabel("Reconstruction Error")
plt.ylabel("Samples")
plt.show()

print("Anomaly Detection Completed Successfully!")